# SIH26054 — SOTA EDL Training on Kaggle (+ W&B)

Clone this repo on Kaggle and train the EvidentialFaultClassifier + OOD detector.
No external datasets needed — data is physics-simulated.

**Prereqs**: `Settings → Accelerator: GPU T4 x2`, `Internet: ON`.

**W&B (optional but recommended)**: Create a W&B account at wandb.ai → API key → Kaggle `Add-ons → Secrets → Add Secret` name `WANDB_API_KEY` (paste key), check `Attach to this notebook`. Models & metrics auto-push to W&B; you can download from there even after Kaggle session ends.

Replace `REPO_URL` in Cell 1 with your GitHub URL.


In [ ]:
# Cell 1 — Clone repo
import os, pathlib
REPO_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"
BRANCH = "master"

!rm -rf /kaggle/working/sih
!git clone {REPO_URL} /kaggle/working/sih
%cd /kaggle/working/sih
!git checkout {BRANCH}
!ls -la
!cat backend/config.py | head -40

In [ ]:
# Cell 1b — W&B setup (optional) — run BEFORE Cell 2 if you want cloud push
# 1) In Kaggle UI: Add-ons → Secrets → New secret → Name: WANDB_API_KEY → Value: your wandb.ai API key
# 2) Ensure Secrets is attached to this notebook (toggle).
# 3) Then run this cell. If you skip it, training runs fine without W&B, or in offline mode.
import os, pathlib
!pip install -q wandb
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("WANDB_API_KEY loaded from Kaggle Secrets:", "***" + os.environ["WANDB_API_KEY"][-4:] if os.environ["WANDB_API_KEY"] else "EMPTY")
except Exception as e:
    print("Kaggle Secrets not available (local run or secret not attached):", e)
    print("If WANDB_API_KEY is set in env manually, it will be used. Otherwise wandb will run in offline/disabled mode.")

# Optional: set W&B project/entity/name here (or pass via --wandb-* flags in Cell 4)
os.environ["WANDB_PROJECT"] = "sih26054-digital-twin"
# os.environ["WANDB_ENTITY"] = "your-team"  # uncomment if using team entity
print("W&B project:", os.environ.get("WANDB_PROJECT"))
print("W&B mode: will be 'online' if WANDB_API_KEY set, else offline — see Cell 4 flags")

In [ ]:
# Cell 2 — Install deps + verify GPU & simulator
%cd /kaggle/working/sih
!pip install -q filterpy onnx onnxruntime pyarrow tqdm wandb
# Editable install is optional — training imports via PYTHONPATH (cwd) even if this fails.
# Fixed pyproject: requires-python >=3.10 + hatchling wheel packages. This should now succeed, but we handle gracefully.
!pip install -q -e . || echo "[warn] pip -e . failed — adding repo to PYTHONPATH instead"
import sys; sys.path.insert(0, "/kaggle/working/sih")
import os; os.environ["PYTHONPATH"] = "/kaggle/working/sih:" + os.environ.get("PYTHONPATH", "")

import torch
print("torch", torch.__version__, "cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

import wandb
print("wandb", wandb.__version__)
print("WANDB_API_KEY set:", bool(os.environ.get("WANDB_API_KEY")))
if not os.environ.get("WANDB_API_KEY"):
    print("[warn] WANDB_API_KEY not set — W&B is compulsory (online). Run Cell 1b or add Kaggle Secret WANDB_API_KEY")

from backend.simulator.engine_simulator import EngineSimulator
sim = EngineSimulator(seed=0)
frame = sim.step()
print("Simulator OK — rpm:", round(frame.rpm,1), "cht:", [round(x,1) for x in frame.cht_c])


In [ ]:
# Cell 3 — Generate training data (physics-simulated)
# Quick smoke: ~1-2 min, 1k windows. Use for first verification.
# For full SOTA: comment quick and uncomment one of the full/default lines.
%cd /kaggle/working/sih
!python scripts/generate_all_data.py --quick
# !python scripts/generate_all_data.py              # default: 10 min missions, ~12k windows, 4-6 min
# !python scripts/generate_all_data.py --full       # full: 60 min missions, ~30k windows, 8-12 min

!ls -lh data/training/
!ls -lh data/models/residual_stats.npz
!cat data/training/generation_stats.json
import numpy as np, json, pathlib
d = np.load("data/training/windows.npz")
print("X", d["X"].shape, "y", d["y"].shape, "bincount", np.bincount(d["y"]))
from backend.config import FAULT_CLASSES
print("classes:", FAULT_CLASSES)

In [ ]:
# Cell 3ALT — (Optional) Skip generation if you uploaded windows.npz as Kaggle Dataset
# If you attached a dataset named sih-windows containing windows.npz + residual_stats.npz:
# !mkdir -p data/training data/models
# !cp /kaggle/input/sih-windows/windows.npz data/training/windows.npz
# !cp /kaggle/input/sih-windows/residual_stats.npz data/models/residual_stats.npz
# !ls -lh data/training/ data/models/
print("Skip — only run if you attached a pre-generated dataset")

In [ ]:
# Cell 4 — Train Evidential Model (SOTA) — W&B COMPULSORY
%cd /kaggle/working/sih
# W&B is now compulsory — training fails if WANDB_API_KEY not set (see Cell 1b).
# Quick smoke (3 epochs, ~30s) — auto-pushes to W&B project sih26054-digital-twin
!python -m backend.ml.training.train_evidential --quick --num-workers 0 --wandb-project sih26054-digital-twin --wandb-tags quick sota kaggle

# --- Debugging offline (if key missing, but W&B still required locally) ---
# !python -m backend.ml.training.train_evidential --quick --num-workers 0 --wandb-mode offline --wandb-project sih26054-digital-twin

# --- Full SOTA: 60 epochs, early stopping, AMP on GPU — compulsory W&B push ---
# !python -m backend.ml.training.train_evidential --epochs 60 --batch-size 64 --num-workers 0 --wandb-project sih26054-digital-twin --wandb-tags full sota

# --- Tuned example (W&B compulsory) ---
# !python -m backend.ml.training.train_evidential --epochs 60 --lr 1e-3 --mixup 0.2 --scheduler cosine --num-workers 0 --wandb-project sih26054-digital-twin


In [ ]:
# Cell 4b — (Alternative) Train via wrapper script (W&B compulsory)
# !python scripts/train_models.py --quick --wandb-project sih26054-digital-twin
print("Use only if you prefer scripts/train_models.py wrapper — W&B still compulsory")


In [ ]:
# Cell 5 — Calibrate OOD detector (Mahalanobis)
# Auto-run at end of training; re-run standalone if needed
%cd /kaggle/working/sih
!python -m backend.ml.training.calibrate_ood --threshold 99.0
import numpy as np
stats = np.load("data/models/ood_stats.npz")
print("threshold", float(stats["threshold"]), "mean shape", stats["mean"].shape)
if "pca_components" in stats:
    print("PCA components", stats["pca_components"].shape)

In [ ]:
# Cell 6 — Validate & inspect (+ W&B table already logged)
%cd /kaggle/working/sih
!python scripts/validate_models.py

import numpy as np
from backend.ml.inference import EvidentialModelInference
infer = EvidentialModelInference()
print("Inference loaded:", infer.loaded)
d = np.load("data/training/windows.npz")
X, y = d["X"], d["y"]
for i in [0, len(y)//3, len(y)//2, -1]:
    out = infer.infer(X[i])
    print(f"true={y[i]} pred={out['predicted_label']} epistemic={out['epistemic_uncertainty']:.3f} conf={out['confidence']:.3f} fault_prob={out['fault_probability']:.3f}")

import json, pathlib
try:
    import matplotlib.pyplot as plt
    hist = json.loads(pathlib.Path("data/models/training_history.json").read_text())
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    axes[0].plot([h["train_loss"] for h in hist], label="train_loss")
    axes[0].plot([h["val_loss"] for h in hist], label="val_loss")
    axes[0].legend(); axes[0].set_title("Loss")
    axes[1].plot([h["val_acc"] for h in hist], label="val_acc")
    axes[1].plot([h["val_ece"] for h in hist], label="val_ece")
    axes[1].legend(); axes[1].set_title("Acc / ECE")
    plt.show()
except Exception as e:
    print("Plot skipped:", e)
    import json; print(json.dumps(hist[-1], indent=2) if 'hist' in locals() else "no hist")

In [ ]:
# Cell 7 — W&B artifacts verification
%cd /kaggle/working/sih
!ls -lh data/models/
!cat data/models/val_metrics.json
!cat data/models/test_metrics.json

# If W&B online, artifact URL is printed in Cell 4 logs (wandb Run URL). Find it at wandb.ai → your project → Runs → Artifacts.
# List local W&B files for offline sync
!ls -lh wandb/ 2>&1 | head -20
!find wandb -name "*.wandb" 2>&1 | head -5
print("If you ran with --wandb-mode offline, sync later via: wandb sync wandb/latest-run/")

In [ ]:
# Cell 8 — Save artifacts for download (still useful even with W&B)
%cd /kaggle/working/sih
!ls -lh data/models/
!zip -r /kaggle/working/sih_models.zip data/models/
print("Zip at /kaggle/working/sih_models.zip — Download via Kaggle Output panel (right sidebar)")
!ls -lh /kaggle/working/sih_models.zip
!echo "Files to commit back to repo (also in W&B artifact):"
!ls -1 data/models/evidential_model.onnx data/models/evidential_model.pt data/models/ood_stats.npz data/models/residual_stats.npz 2>&1

# Optional: also upload zip as W&B artifact explicitly
try:
    import wandb
    if wandb.run is not None:
        art = wandb.Artifact("sih-models-zip", type="dataset")
        art.add_file("/kaggle/working/sih_models.zip")
        wandb.run.log_artifact(art)
        print("Zip also logged to W&B")
except Exception as e:
    print("W&B zip log skipped:", e)

### Optional: Push back to GitHub from Kaggle
Add a Kaggle Secret `GITHUB_TOKEN` (Settings → Secrets) then run:
```python
!git config --global user.email "kaggle@kaggle.com"
!git config --global user.name "kaggle"
!git add data/models/evidential_model.onnx data/models/ood_stats.npz data/models/residual_stats.npz data/models/training_history.json
!git commit -m "kaggle: add trained SOTA models"
!git push https://$GITHUB_TOKEN@github.com/<USER>/<REPO>.git HEAD:master
```
Or pull from W&B: `wandb artifact get <entity>/<project>/sih-edl-<run_id>:latest --root data/models/` or download zip from Kaggle Output. Or just `pip install wandb && wandb login` locally then `wandb artifact get`.
